# 01 Data Exploration — Verifying Data Availability

Check which data sources can be fetched reliably via yfinance and pandas-datareader.

| Category | Contents |
|----------|----------|
| Target | Nikkei 225 (^N225) |
| Macro (futures & indices) | Nikkei 225 futures, S&P 500, NASDAQ, Dow Jones |
| Macro (FX) | USD/JPY, EUR/JPY, GBP/JPY, CNY/JPY |
| Macro (commodities) | WTI crude oil, gold, silver, copper |
| Macro (bonds) | US Treasuries (2Y/10Y/30Y), JGB 10Y (multiple sources) |
| Micro (Nikkei 225 constituents) | Fast Retailing, SoftBank, Tokyo Electron, KDDI, Shin-Etsu |

## 0. Environment Setup (Google Colab)

Run this cell only when executing on Google Colab.

In [ ]:
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())

if IN_COLAB:
    # Clone repository
    !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git
    %cd Nikkei_Analysis

    # Mount Google Drive for data persistence
    from google.colab import drive
    drive.mount('/content/drive')

    # Link data/ to Google Drive
    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists('data'):
        os.symlink(DRIVE_DATA, 'data')

    # Install dependencies
    !pip install -q yfinance pandas-ta pandas-datareader

    # Add src/ to import path
    import sys
    sys.path.insert(0, '/content/Nikkei_Analysis')

    print('Colab setup complete.')
else:
    print('Running in local environment.')

In [ ]:
import warnings
import os
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.4f}'.format)

PERIOD   = '2y'
INTERVAL = '1d'
DATA_RAW = 'data/raw'

END   = datetime.today()
START = END - timedelta(days=365 * 2)

## 1. Target: Nikkei 225

In [ ]:
nikkei = yf.download('^N225', period=PERIOD, interval=INTERVAL)
print(f'Rows   : {len(nikkei)}')
print(f'Period : {nikkei.index[0].date()} to {nikkei.index[-1].date()}')
print(f'Missing: {nikkei.isnull().sum().to_dict()}')
display(nikkei.tail())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
nikkei['Close'].plot(ax=axes[0], title='Nikkei 225 Close')
nikkei['Volume'].plot(ax=axes[1], title='Volume', color='orange')
axes[0].set_ylabel('Price (JPY)')
axes[1].set_ylabel('Volume')
plt.tight_layout()
plt.show()

## 2. Macro: Futures & Equity Indices

In [ ]:
macro_index_tickers = {
    'Nikkei225_Futures': 'NKD=F',
    'SP500':             '^GSPC',
    'NASDAQ':            '^IXIC',
    'DowJones':          '^DJI',
    'VIX':               '^VIX',
}

results_index = {}
for name, ticker in macro_index_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_index[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<25} ticker={ticker:<10} rows={len(df):4d}  close_na={na}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_index.items():
    if len(df) > 0:
        (df['Close'] / df['Close'].iloc[0] * 100).plot(ax=ax, label=name)
(nikkei['Close'] / nikkei['Close'].iloc[0] * 100).plot(
    ax=ax, label='Nikkei225', linewidth=2, linestyle='--', color='black')
ax.set_title('Equity Indices — Normalized (base=100)')
ax.set_ylabel('Index (base=100)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Macro: FX Rates

In [ ]:
fx_tickers = {
    'USD_JPY': 'JPY=X',
    'EUR_JPY': 'EURJPY=X',
    'GBP_JPY': 'GBPJPY=X',
    'CNY_JPY': 'CNYJPY=X',
}

results_fx = {}
for name, ticker in fx_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_fx[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<12} ticker={ticker:<12} rows={len(df):4d}  close_na={na}")

In [ ]:
n = len(results_fx)
fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), sharex=True)
for ax, (name, df) in zip(axes, results_fx.items()):
    if len(df) > 0:
        df['Close'].plot(ax=ax, title=name)
        ax.set_ylabel('Rate (JPY)')
plt.tight_layout()
plt.show()

## 4. Macro: Commodities

In [ ]:
commodity_tickers = {
    'WTI_Crude': 'CL=F',
    'Gold':      'GC=F',
    'Silver':    'SI=F',
    'Copper':    'HG=F',
}

results_commodity = {}
for name, ticker in commodity_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_commodity[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<12} ticker={ticker:<8} rows={len(df):4d}  close_na={na}")

In [ ]:
n = len(results_commodity)
fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), sharex=True)
for ax, (name, df) in zip(axes, results_commodity.items()):
    if len(df) > 0:
        df['Close'].plot(ax=ax, title=name)
        ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

## 5. Macro: US Treasury Yields

In [ ]:
bond_tickers = {
    'US_2Y':  '^IRX',
    'US_10Y': '^TNX',
    'US_30Y': '^TYX',
}

results_bond = {}
for name, ticker in bond_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_bond[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<8} ticker={ticker:<8} rows={len(df):4d}  close_na={na}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_bond.items():
    if len(df) > 0:
        df['Close'].plot(ax=ax, label=name)
ax.set_title('US Treasury Yields')
ax.set_ylabel('Yield (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Macro: JGB (Japanese Government Bond) Yields

yfinance does not provide reliable JGB yield data.
We test three alternative sources below.

| Source | Series | Frequency | Notes |
|--------|--------|-----------|-------|
| stooq (via pandas-datareader) | `10lj.b` | Daily | Free, no API key |
| FRED (via pandas-datareader) | `IRLTLT01JPM156N` | Monthly | Free, no API key |
| Bank of Japan (web scrape) | Statistics page | Daily | Official source, scraping required |

In [ ]:
results_jgb = {}

# --- Attempt 1: stooq (daily) ---
try:
    df_stooq = web.DataReader('10lj.b', 'stooq', start=START, end=END)
    df_stooq = df_stooq.sort_index()
    results_jgb['stooq_10Y'] = df_stooq
    print(f'OK | stooq  rows={len(df_stooq)}  na={df_stooq["Close"].isnull().sum()}')
except Exception as e:
    results_jgb['stooq_10Y'] = pd.DataFrame()
    print(f'NG | stooq  error: {e}')

# --- Attempt 2: FRED (monthly) ---
try:
    df_fred = web.DataReader('IRLTLT01JPM156N', 'fred', start=START, end=END)
    df_fred.columns = ['Close']
    results_jgb['FRED_10Y_monthly'] = df_fred
    print(f'OK | FRED   rows={len(df_fred)}  na={df_fred["Close"].isnull().sum()}  (monthly)')
except Exception as e:
    results_jgb['FRED_10Y_monthly'] = pd.DataFrame()
    print(f'NG | FRED   error: {e}')

In [ ]:
# --- Attempt 3: Bank of Japan official CSV ---
# BOJ publishes daily JGB yields as a downloadable CSV.
# URL pattern: https://www.boj.or.jp/statistics/market/interest/eyr/de.csv
# Column 'IR6020.Q' contains the 10Y JGB yield.
try:
    import requests
    from io import StringIO

    url = 'https://www.boj.or.jp/statistics/market/interest/eyr/de.csv'
    response = requests.get(url, timeout=10)
    response.encoding = 'shift_jis'

    df_boj_raw = pd.read_csv(StringIO(response.text), skiprows=8, header=0)
    df_boj_raw.columns = df_boj_raw.columns.str.strip()

    date_col = df_boj_raw.columns[0]
    df_boj_raw[date_col] = pd.to_datetime(df_boj_raw[date_col], errors='coerce')
    df_boj_raw = df_boj_raw.dropna(subset=[date_col]).set_index(date_col)

    # 10Y JGB yield column (IR6020.Q)
    jgb_col = 'IR6020.Q'
    if jgb_col in df_boj_raw.columns:
        df_boj = df_boj_raw[[jgb_col]].rename(columns={jgb_col: 'Close'})
        df_boj['Close'] = pd.to_numeric(df_boj['Close'], errors='coerce')
        df_boj = df_boj.loc[START:END]
        results_jgb['BOJ_10Y_daily'] = df_boj
        print(f'OK | BOJ    rows={len(df_boj)}  na={df_boj["Close"].isnull().sum()}  (daily)')
        display(df_boj.tail())
    else:
        available_cols = df_boj_raw.columns.tolist()[:10]
        print(f'NG | BOJ    column {jgb_col} not found. Available: {available_cols}')
        results_jgb['BOJ_10Y_daily'] = pd.DataFrame()

except Exception as e:
    results_jgb['BOJ_10Y_daily'] = pd.DataFrame()
    print(f'NG | BOJ    error: {e}')

In [ ]:
# Plot all successful JGB sources
fig, ax = plt.subplots(figsize=(12, 4))
plotted = False
for name, df in results_jgb.items():
    if len(df) > 0 and 'Close' in df.columns:
        df['Close'].dropna().plot(ax=ax, label=name)
        plotted = True
if plotted:
    ax.set_title('JGB 10Y Yield — Source Comparison')
    ax.set_ylabel('Yield (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No JGB data available from any source.')

## 7. Micro: Nikkei 225 Major Constituents

In [ ]:
micro_tickers = {
    'FastRetailing':  '9983.T',
    'SoftBank':       '9984.T',
    'Tokyo_Electron': '8035.T',
    'KDDI':           '9433.T',
    'ShinEtsu_Chem':  '4063.T',
}

results_micro = {}
for name, ticker in micro_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_micro[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<20} ticker={ticker:<8} rows={len(df):4d}  close_na={na}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_micro.items():
    if len(df) > 0:
        (df['Close'] / df['Close'].iloc[0] * 100).plot(ax=ax, label=name)
ax.set_title('Nikkei 225 Major Constituents — Normalized (base=100)')
ax.set_ylabel('Index (base=100)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
all_results = {
    'Nikkei225':                nikkei,
    **{f'[Index] {k}':     v for k, v in results_index.items()},
    **{f'[FX] {k}':        v for k, v in results_fx.items()},
    **{f'[Commodity] {k}': v for k, v in results_commodity.items()},
    **{f'[Bond_US] {k}':   v for k, v in results_bond.items()},
    **{f'[JGB] {k}':       v for k, v in results_jgb.items()},
    **{f'[Micro] {k}':     v for k, v in results_micro.items()},
}

summary = []
for name, df in all_results.items():
    if len(df) > 0 and 'Close' in df.columns:
        summary.append({
            'series':    name,
            'available': True,
            'rows':      len(df),
            'start':     df.index[0].date(),
            'end':       df.index[-1].date(),
            'close_na':  int(df['Close'].isnull().sum()),
            'na_pct':    round(df['Close'].isnull().mean() * 100, 1),
        })
    else:
        summary.append({'series': name, 'available': False,
                        'rows': 0, 'start': None, 'end': None,
                        'close_na': None, 'na_pct': None})

summary_df = pd.DataFrame(summary)
display(summary_df)

In [ ]:
# Merge all available Close prices into a single DataFrame
close_df = pd.DataFrame({'Nikkei225': nikkei['Close'].squeeze()})

all_close_sources = {
    **results_index, **results_fx, **results_commodity,
    **results_bond, **results_jgb, **results_micro
}
for name, df in all_close_sources.items():
    if len(df) > 0 and 'Close' in df.columns:
        close_df[name] = df['Close'].squeeze()

print(f'Shape : {close_df.shape}')
print('\nMissing rate (%):')
print((close_df.isnull().mean() * 100).round(1).to_string())
display(close_df.tail())

## 9. Save Raw Data

In [ ]:
os.makedirs(DATA_RAW, exist_ok=True)
save_path = f'{DATA_RAW}/close_all.csv'
close_df.to_csv(save_path)
print(f'Saved: {save_path}  shape={close_df.shape}')

## 10. Next Steps

Based on the summary above:

- **Failed sources** — find alternative tickers or exclude
- **JGB** — adopt the best-performing source above; if BOJ daily works, use it; otherwise fall back to FRED monthly with forward-fill
- **High missing rate** — decide on imputation strategy (forward-fill, interpolation, or drop)

Next: implement `src/data/fetcher.py` to encapsulate all data fetching logic as reusable functions.